# Plan 003.3c — Available tag vocabulary

A public, synthetic walkthrough of Archiver's catalog-wide active tag vocabulary API and
`catalog tags available` CLI.

This tutorial is for developers and users who understand Archiver's content identity and
content-level tag concepts.

**Prerequisites**

- Run the notebook from the repository root in the project environment.
- Know that tags describe SHA-256 content identity, not individual paths.
- Know that an assertion is active until it is retracted and records user or system provenance.

**Learning goals**

By the end, you will be able to:

1. distinguish distinct-content counts from active assertion counts;
2. inspect the complete active vocabulary without requiring current paths;
3. filter counts by provenance and tag-name regular expressions;
4. sort and bound API and CLI results without losing complete totals;
5. explain why retracted-only names disappear while orphaned tagged content remains.

## Outline

1. Create and scan disposable synthetic files.
2. Seed active user and system assertions plus one retracted assertion.
3. Inspect vocabulary counts through the typed API.
4. Use direct `!archiver` commands for default, bounded, regex, provenance, and reverse views.
5. Remove a path and confirm its content's active tag remains in the vocabulary.
6. Try an exercise, review pitfalls, and clean up.

## Semantic boundary

One vocabulary row represents one canonical tag name with at least one active assertion
anywhere in the catalog.

- `Content` counts distinct content identities carrying the tag.
- `Assertions` counts active provenance rows.
- Duplicate paths do not increase `Content`.
- User and system assertions of the same tag on one content identity increase
  `Assertions` but still count that content once.
- Current paths are irrelevant: tagged content remains part of the vocabulary after every
  observed path disappears.

## 1. Create disposable synthetic content

Two photo paths contain identical bytes, giving them one shared content identity. The old
report will later be removed to demonstrate that vocabulary membership is catalog-wide.

In [ ]:
from __future__ import annotations

from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory

from archiver import AvailableTagSearch, Catalog, TagProvenance

workspace = TemporaryDirectory(prefix="archiver-plan-0033c-")
base = Path(workspace.name)
root = base / "demo-root"
(root / "photos").mkdir(parents=True)
(root / "backups").mkdir()
(root / "notes").mkdir()
(root / "old").mkdir()

shared_photo_bytes = b"same synthetic photo"
(root / "photos" / "day-01.jpg").write_bytes(shared_photo_bytes)
(root / "backups" / "day-01-copy.jpg").write_bytes(shared_photo_bytes)
(root / "photos" / "group.jpg").write_bytes(b"different synthetic group photo")
(root / "notes" / "readme.txt").write_bytes(b"synthetic notes")
(root / "old" / "report.txt").write_bytes(b"historical synthetic report")
(root / "loose.bin").write_bytes(b"untagged bytes")

print(f"Disposable root: {root}")
for source_path in sorted(path for path in root.rglob("*") if path.is_file()):
    print(" ", source_path.relative_to(root).as_posix())

## 2. Initialize and scan with the CLI

Every CLI example uses direct IPython `!archiver` syntax. The temporary root contains no
pre-existing catalog.

In [ ]:
# Create the in-root schema-version-2 catalog.
!archiver catalog init "{root}"

In [ ]:
# Record the first successful current state without interactive progress output.
!archiver catalog scan "{root}" --no-progress

## 3. Seed provenance-aware assertions

Plan 003.3c is query-only, so this setup uses the Plan 003.2 mutation API. The same `family`
tag is asserted by both user and system provenance on the shared photo content. Its two
paths still represent one content identity.

The `obsolete` assertion is immediately retracted. It remains historical metadata but must
not appear in the active vocabulary.

In [ ]:
catalog_path = root / ".archiver" / "catalog.sqlite"
catalog = Catalog.open(catalog_path)

user = TagProvenance("user", "notebook", "1.0")
system = TagProvenance("system", "demo-classifier", "1.0", "synthetic-rules")

shared_photo = catalog.content_for_path(root, PurePosixPath("photos/day-01.jpg"))
group_photo = catalog.content_for_path(root, PurePosixPath("photos/group.jpg"))
notes = catalog.content_for_path(root, PurePosixPath("notes/readme.txt"))
old_report = catalog.content_for_path(root, PurePosixPath("old/report.txt"))

for tag in ("family", "favorite"):
    catalog.add_content_tag(shared_photo, tag, user)
for tag in ("family", "picture", "trip:us"):
    catalog.add_content_tag(shared_photo, tag, system)

for tag in ("family", "trip:us"):
    catalog.add_content_tag(group_photo, tag, user)
catalog.add_content_tag(group_photo, "picture", system)

catalog.add_content_tag(notes, "text", user)
catalog.add_content_tag(notes, "obsolete", user)
catalog.retract_content_tag(notes, "obsolete", user)
catalog.add_content_tag(old_report, "remember", user)

shared_paths = [
    observation.relative_path.as_posix()
    for observation in catalog.find_by_content(root, shared_photo)
]
print("Shared photo digest:", shared_photo.digest)
print("Shared photo paths:", shared_paths)
print("Schema version:", catalog.schema_version)

## 4. Query the complete vocabulary through the typed API

`AvailableTagSearch.total_matches` describes all matching active names. Its `tags` tuple is
the bounded projection. Each `TagUsage` row separates distinct content from assertion and
provenance counts.

In [ ]:
def show_usage(label: str, result: AvailableTagSearch) -> None:
    print(f"{label}: {result.total_matches} matching active tags; {len(result.tags)} rows shown")
    print(f"  {'Tag':<12} {'Content':>7} {'Assertions':>10} {'User':>5} {'System':>6}")
    for usage in result.tags:
        print(
            f"  {usage.tag:<12} {usage.content_count:>7} "
            f"{usage.assertion_count:>10} {usage.user_assertion_count:>5} "
            f"{usage.system_assertion_count:>6}"
        )


all_usage = catalog.search_available_tags()
show_usage("all active vocabulary", all_usage)

by_tag = {usage.tag: usage for usage in all_usage.tags}
assert all_usage.total_matches == 6
assert list(by_tag) == ["family", "favorite", "picture", "remember", "text", "trip:us"]
assert by_tag["family"].content_count == 2
assert by_tag["family"].assertion_count == 3
assert by_tag["family"].user_assertion_count == 2
assert by_tag["family"].system_assertion_count == 1
assert "obsolete" not in by_tag

The shared photo has two current paths, but `family` counts its bytes once. Its user and
system assertions are separate active rows, which is why `family` has three assertions
across two content identities.

In [ ]:
# The 3 files under tag==family (2 content ids)
!archiver catalog files "{root}" --tag family 
#--provenance system 

## 5. Bound rows while preserving the complete total

Assertion sorting defaults to descending. Equal assertion counts use tag name ascending as
the deterministic tie-breaker. The SQL row limit bounds the returned tuple, while
`total_matches` stays complete.

In [ ]:
bounded = catalog.search_available_tags(sort_by="assertions", limit=3)
show_usage("top assertion counts", bounded)

assert bounded.total_matches == 6
assert len(bounded.tags) == 3
assert [usage.tag for usage in bounded.tags] == ["family", "picture", "trip:us"]

## 6. Filter by provenance

Provenance controls vocabulary inclusion and every count column. Under `system`, a tag with
only user assertions disappears, user counts are zero, and content is deduplicated within
the remaining system assertions.

In [ ]:
system_usage = catalog.search_available_tags(provenance="system", sort_by="assertions")
show_usage("system vocabulary", system_usage)

assert [usage.tag for usage in system_usage.tags] == ["picture", "family", "trip:us"]
assert all(usage.user_assertion_count == 0 for usage in system_usage.tags)

## 7. Filter names with Python regular expressions

The API and CLI use Python `re.search` over the complete canonical name. Matching is
case-sensitive unless the expression supplies an inline flag.

In [ ]:
regex_usage = catalog.search_available_tags(
    name_regex=r"(?i)^(FAMILY|PICTURE)$",
    sort_by="content",
)
show_usage("case-insensitive family or picture", regex_usage)

assert [usage.tag for usage in regex_usage.tags] == ["family", "picture"]

## 8. Use `catalog tags available` directly

The default CLI view shows all active names alphabetically, complete totals, the effective
sort direction, and effective provenance.

In [ ]:
# Default: name ascending, all provenance, 20-row bound.
!archiver catalog tags available "{root}"

Sorting by assertions and limiting to three rows changes only the displayed projection.
The summary still reports all six matching active names.

In [ ]:
# Show the three highest active assertion counts.
!archiver catalog tags available "{root}" --sort assertions --limit 3

Regex and provenance filters compose. The original regex source and effective provenance
are printed with the result.

In [ ]:
# Restrict both inclusion and counts to system assertions, then filter names.
!archiver catalog tags available "{root}" --provenance system --regex "(?i)^(FAMILY|PICTURE|TRIP:US)$" --sort assertions

Reversing name sort produces descending names. This is different from count sorts, where
reverse changes only the primary numeric direction and alphabetical ties remain ascending.

In [ ]:
# Reverse the primary name order and bound the displayed rows.
!archiver catalog tags available "{root}" --sort name --reverse --limit 4

Invalid regex is a CLI usage error. Argument parsing rejects it before catalog-path
resolution or database opening.

In [ ]:
# Expected usage error: the character class is not closed.
!archiver catalog tags available "{root}" --regex "["

## 9. Remove the only current path for tagged content

Removing this disposable source path and scanning again changes current observations only.
The `remember` assertion remains attached to the old report's content identity.

In [ ]:
old_report_path = root / "old" / "report.txt"
old_report_path.unlink()
print("Removed disposable path:", old_report_path.relative_to(root).as_posix())

In [ ]:
# Refresh current observations after the disposable path disappears.
!archiver catalog scan "{root}" --no-progress

In [ ]:
# Vocabulary is catalog-wide, so the orphaned content's active tag still appears.
!archiver catalog tags available "{root}" --regex "^remember$"

In [ ]:
orphaned_usage = catalog.search_available_tags(name_regex=r"^remember$")
assert orphaned_usage.total_matches == 1
assert orphaned_usage.tags[0].content_count == 1
assert catalog.find_by_content(root, old_report) == []
show_usage("orphaned tagged content", orphaned_usage)

The retracted-only `obsolete` name remains absent:

In [ ]:
!archiver catalog tags available "{root}" --regex "^obsolete$"

## Exercise

Before running the answer cell, predict the output names for:

- provenance `system`;
- sort by assertions descending;
- limit 2.

How many total system tag names match, and why does `picture` sort first?

In [ ]:
# Exercise answer: picture has two system assertions; family and trip:us have one each.
!archiver catalog tags available "{root}" --provenance system --sort assertions --limit 2

In [ ]:
exercise = catalog.search_available_tags(
    provenance="system",
    sort_by="assertions",
    limit=2,
)
assert exercise.total_matches == 3
assert [usage.tag for usage in exercise.tags] == ["picture", "family"]
show_usage("exercise answer", exercise)

## Pitfalls and extensions

- **Do not count paths as content.** Duplicate paths for the same bytes contribute one
  distinct content identity per tag.
- **Do not infer complete totals from bounded rows.** Compare `total_matches` with
  `len(tags)` or read the CLI summary.
- **Do not treat provenance as a display-only filter.** It controls inclusion and all four
  count columns.
- **Do not expect retracted-only names to appear.** Retraction preserves history but removes
  the assertion from the active vocabulary.
- **Do not expect tags to disappear with paths.** Vocabulary membership is catalog-wide and
  independent of current scans.

Optional extension: add another synthetic system source asserting `family` on the group
photo and compare the content and assertion counts before and after.

## Cleanup

In [ ]:
catalog.close()
workspace.cleanup()
print("Temporary catalog and synthetic files removed.")